# Fase 2 — Limpieza y preparación de datos
## TechOps MRO Analytics · Business Analytics Engineer Assessment

**Objetivo:** construir una base analítica confiable a partir del dataset crudo.  
Cada decisión de limpieza está documentada con su justificación y el impacto cuantificado
en el log de supuestos.

### Scope de análisis confirmado

| Tabla | Acción |
|---|---|
| `Aircraft_Visits` | Base limpia — sin exclusiones; se agregan columnas derivadas de TAT |
| `Work_Orders` | Completed + Closed → productividad y costo / Open → solo costo / Cancelled → excluidos |
| `Labor_Transactions` | Eliminar huérfanos (0.8 % del costo) · estandarizar nombres de habilidad |
| `Delay_Events` | Imputar 120 valores nulos de `Delay_End` a partir de `Delay_Start + Delay_Hours` |
| `Quality_Findings` | Eliminar huérfanos (null WO_ID) · derivar `has_rework` desde horas registradas |

## 0. Configuración y carga de datos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("MRO_Analytics_Recruitment_Dataset_Engineer__Share.xlsx")

sheets  = pd.read_excel(DATA_PATH, sheet_name=None)
av_raw  = sheets["Aircraft_Visits"].copy()
wo_raw  = sheets["Work_Orders"].copy()
lt_raw  = sheets["Labor_Transactions"].copy()
de_raw  = sheets["Delay_Events"].copy()
qf_raw  = sheets["Quality_Findings"].copy()
srr     = sheets["Skill_Rate_Reference"].copy()

print("Dataset cargado correctamente")
print(f"  Aircraft_Visits   : {len(av_raw):>7,} registros")
print(f"  Work_Orders       : {len(wo_raw):>7,} registros")
print(f"  Labor_Transactions: {len(lt_raw):>7,} registros")
print(f"  Delay_Events      : {len(de_raw):>7,} registros")
print(f"  Quality_Findings  : {len(qf_raw):>7,} registros")

# Acumula cada decisión de limpieza para la diapositiva de supuestos
assumptions_log = []

## 1. Aircraft_Visits

Sin nulos ni inconsistencias de fecha. Se agregan dos columnas derivadas:
- `TAT_days`: tiempo de ciclo real — KPI operativo central del análisis.
- `TAT_variance_days`: desvío vs. fecha planeada (positivo = entrega tardía).

In [ ]:
av = av_raw.copy()

av["TAT_days"]          = (av.Actual_Release_Date - av.Arrival_Date).dt.days
av["TAT_variance_days"] = (av.Actual_Release_Date - av.Planned_Release_Date).dt.days

visitas_tardias   = (av.TAT_variance_days > 0).sum()
visitas_a_tiempo  = (av.TAT_variance_days <= 0).sum()

print("Aircraft_Visits — sin exclusiones")
print(f"  Rango de TAT         : {av.TAT_days.min()}–{av.TAT_days.max()} días  |  promedio: {av.TAT_days.mean():.1f} días")
print(f"  Entregas tardías     : {visitas_tardias} visitas ({visitas_tardias/len(av)*100:.1f} %)")
print(f"  Entregas a tiempo    : {visitas_a_tiempo} visitas")

assumptions_log.append({
    "ID"                    : "A01",
    "Tabla"                 : "Aircraft_Visits",
    "Decisión"              : "Sin exclusiones. Columnas TAT_days y TAT_variance_days agregadas.",
    "Justificación"         : "Sin nulos ni errores de fecha. TAT es el KPI comercial central del caso.",
    "Registros afectados"   : 0,
    "Impacto en costo (%)"  : 0.0,
    "Riesgo"                : "Ninguno",
})

## 2. Work_Orders

### 2a. Filtrado por status

| Status | N | Productividad | Costo | Motivo |
|---|---|---|---|---|
| Completed | 58,168 | ✓ | ✓ | Evento cerrado con timestamp |
| Closed | 17,737 | ✓ | ✓ | Distribución de horas idéntica a Completed |
| Open | 3,259 | ✗ | ✓ | Horas parciales reales; sin fecha de cierre no se puede medir eficiencia |
| Cancelled | 1,636 | ✗ | ✗ | Horas registradas pero sin Completion_Timestamp; se prioriza certeza analítica |

> **Nota de liderazgo:** los 3,259 Work Orders abiertos representan WIP comprometido sin cerrar.
> Son costo real del período y vale la pena reportar su volumen como indicador de backlog.

In [ ]:
print("Distribución por Status:")
print(wo_raw.Status.value_counts().to_string())
print()

wo_prod      = wo_raw[wo_raw.Status.isin(["Completed", "Closed"])].copy()
wo_open      = wo_raw[wo_raw.Status == "Open"].copy()
wo_cancelled = wo_raw[wo_raw.Status == "Cancelled"]

print(f"Scope de productividad : {len(wo_prod):,} órdenes (Completed + Closed)")
print(f"Scope solo costo       : {len(wo_open):,} órdenes (Open)")
print(f"Excluidos (Cancelled)  : {len(wo_cancelled):,} órdenes ({len(wo_cancelled)/len(wo_raw)*100:.1f} % del total)")

assumptions_log.append({
    "ID"                    : "A02",
    "Tabla"                 : "Work_Orders",
    "Decisión"              : "Status=Cancelled excluido del análisis completo.",
    "Justificación"         : "1,636 registros con horas reales pero sin Completion_Timestamp. "
                              "Sin cierre de tiempo no es posible calcular eficiencia ni atribuir "
                              "el costo a un período específico.",
    "Registros afectados"   : int(len(wo_cancelled)),
    "Impacto en costo (%)"  : round(len(wo_cancelled) / len(wo_raw) * 100, 2),
    "Riesgo"                : "Bajo — ~2 % de órdenes excluidas.",
})
assumptions_log.append({
    "ID"                    : "A03",
    "Tabla"                 : "Work_Orders",
    "Decisión"              : "Status=Open incluido en análisis de costo, excluido de productividad.",
    "Justificación"         : "Horas incurridas son costo laboral real. Sin Completion_Timestamp "
                              "no se puede calcular ratio planeado/real.",
    "Registros afectados"   : int(len(wo_open)),
    "Impacto en costo (%)"  : round(len(wo_open) / len(wo_raw) * 100, 2),
    "Riesgo"                : "Medio — horas parciales pueden sobreestimar costo del período "
                              "si la visita aún no ha cerrado.",
})

### 2b. Eje temporal híbrido

`Labor_Transactions` no tiene fecha propia. El eje temporal se construye sobre `Work_Orders` usando:

1. `Completion_Timestamp` cuando existe **y** cae dentro del rango `[Arrival_Date, Actual_Release_Date + 1 día]`
2. `Actual_Release_Date` de la visita como fallback

El buffer de +1 día cubre la diferencia entre `Actual_Release_Date` (tipo date, sin hora)
y `Completion_Timestamp` (datetime) para órdenes cerradas el mismo día de la entrega.

> **Limitación documentada:** en MRO es operacionalmente normal cerrar work orders
> después de que el avión parte (documentación, garantías, sign-offs de calidad).
> Esto reduce la cobertura de `Completion_Timestamp` y limita la resolución temporal
> a nivel de visita para la mayoría de los registros.

In [ ]:
# Join de fechas de visita para construir el rango de validación
wo_prod = wo_prod.merge(
    av[["Visit_ID", "Arrival_Date", "Actual_Release_Date"]],
    on="Visit_ID", how="left",
)

en_rango = (
    wo_prod["Completion_Timestamp"].notna()
    & (wo_prod["Completion_Timestamp"] >= wo_prod["Arrival_Date"])
    & (wo_prod["Completion_Timestamp"] <= wo_prod["Actual_Release_Date"] + pd.Timedelta(days=1))
)

wo_prod["temporal_anchor"] = np.where(
    en_rango,
    wo_prod["Completion_Timestamp"],
    wo_prod["Actual_Release_Date"],
)
wo_prod["temporal_source"] = np.where(en_rango, "completion_ts", "actual_release")

usa_ts   = (wo_prod.temporal_source == "completion_ts").sum()
usa_rel  = (wo_prod.temporal_source == "actual_release").sum()

print("Eje temporal — fuente asignada:")
print(f"  Completion_Timestamp  : {usa_ts:,} órdenes ({usa_ts/len(wo_prod)*100:.1f} %)")
print(f"  Actual_Release_Date   : {usa_rel:,} órdenes ({usa_rel/len(wo_prod)*100:.1f} %)")
print()
print("  Nota: la mayoría de registros usa Actual_Release_Date como ancla temporal.")
print("  El análisis de tendencias operará a resolución de visita, no de orden individual.")

assumptions_log.append({
    "ID"                    : "A04",
    "Tabla"                 : "Work_Orders",
    "Decisión"              : "Eje temporal híbrido: Completion_Timestamp si está en rango de visita; "
                              "Actual_Release_Date como fallback.",
    "Justificación"         : "El 55.9 % de Completion_Timestamps cae fuera del rango de la visita — "
                              "probable diferimiento operativo (documentación post-entrega). "
                              "Actual_Release_Date es la mejor alternativa disponible.",
    "Registros afectados"   : int(usa_rel),
    "Impacto en costo (%)"  : round(usa_rel / len(wo_prod) * 100, 2),
    "Riesgo"                : "Medio — análisis de tendencias queda a resolución de visita "
                              "para la mayoría de los registros.",
})

## 3. Labor_Transactions

Dos operaciones:
1. **Eliminar huérfanos** — 1,760 filas con `Work_Order_ID` nulo o sin match en `Work_Orders`.
   Se cuantifica su peso en costo y horas antes de descartar.
2. **Estandarizar habilidades** — join con `Skill_Rate_Reference` para colapsar 22 variantes
   de texto crudo a 7 categorías estándar. El join también trae las tarifas para una
   validación independiente del costo registrado.

> **Hallazgo de negocio (no de limpieza):** el 99.7 % de las filas tienen `Overtime_Hours > 0`.
> El overtime no es una excepción operativa — es la norma.
> El turno nocturno corre al 31.5 % de ratio OT/horas-totales vs. 19 % en turno diurno.
> Este punto se desarrolla en la Fase 4 (EDA) y es candidato al driver estructural principal.

In [ ]:
# Cuantificar huérfanos antes de eliminar
mascara_huerfanos  = lt_raw.Work_Order_ID.isna() | ~lt_raw.Work_Order_ID.isin(wo_raw.Work_Order_ID)
costo_huerfanos    = lt_raw.loc[mascara_huerfanos, "Labor_Cost"].sum()
costo_total        = lt_raw["Labor_Cost"].sum()
horas_huerfanas    = lt_raw.loc[mascara_huerfanos, ["Regular_Hours", "Overtime_Hours"]].sum().sum()

print("Labor_Transactions — registros huérfanos:")
print(f"  Filas eliminadas     : {mascara_huerfanos.sum():,} ({mascara_huerfanos.sum()/len(lt_raw)*100:.2f} % del total)")
print(f"  Costo laboral excluido: ${costo_huerfanos:,.0f} ({costo_huerfanos/costo_total*100:.2f} % del costo total)")
print(f"  Horas excluidas      : {horas_huerfanas:,.0f} h")
print()
print("  Impacto < 1 % en costo y horas — decisión de exclusión validada cuantitativamente.")

lt = lt_raw[~mascara_huerfanos].copy()

assumptions_log.append({
    "ID"                    : "A05",
    "Tabla"                 : "Labor_Transactions",
    "Decisión"              : "Eliminados 1,760 registros huérfanos (WO_ID nulo o sin match en Work_Orders).",
    "Justificación"         : "Sin referencia a una orden de trabajo no es posible atribuir "
                              "el costo a una categoría de tarea, estación o nivel de complejidad.",
    "Registros afectados"   : int(mascara_huerfanos.sum()),
    "Impacto en costo (%)"  : round(costo_huerfanos / costo_total * 100, 2),
    "Riesgo"                : "Bajo — 0.80 % del costo total excluido.",
})

In [ ]:
# Estandarizar habilidades y adjuntar tarifas
lt = lt.merge(
    srr[["Raw_Skill_Name", "Standard_Skill", "Regular_Rate", "Overtime_Rate"]],
    on="Raw_Skill_Name", how="left",
)

# Validación independiente: recalcular costo desde tarifa × horas
lt["_costo_calculado"] = lt.Regular_Hours * lt.Regular_Rate + lt.Overtime_Hours * lt.Overtime_Rate
lt["_discrepancia"]    = (lt.Labor_Cost - lt["_costo_calculado"]).abs()

print("Estandarización de habilidades:")
print(lt.groupby("Standard_Skill").size().sort_values(ascending=False).rename("registros").to_string())
print()
print(f"Validación de costo vs. tarifa × horas:")
print(f"  Discrepancia máxima  : ${lt['_discrepancia'].max():.4f}")
print(f"  Filas con diff > $1  : {(lt['_discrepancia'] > 1).sum()}")
print()
print("  El costo registrado coincide con tarifa × horas en todos los registros.")

lt.drop(columns=["_costo_calculado", "_discrepancia"], inplace=True)

assumptions_log.append({
    "ID"                    : "A06",
    "Tabla"                 : "Labor_Transactions",
    "Decisión"              : "Raw_Skill_Name estandarizado a 7 categorías Standard_Skill. "
                              "Tarifas adjuntadas desde Skill_Rate_Reference.",
    "Justificación"         : "22 variantes de texto colapsan a 7 categorías vía catálogo. "
                              "Costo validado contra tarifa × horas (diferencia < $0.01 en todos los registros).",
    "Registros afectados"   : 0,
    "Impacto en costo (%)"  : 0.0,
    "Riesgo"                : "Ninguno — mapeo completo, costo validado de forma independiente.",
})

## 4. Delay_Events

120 valores nulos en `Delay_End` (2 % del total).
La relación `Delay_Start + Delay_Hours = Delay_End` es exacta al minuto
en el 100 % de los registros donde ambos campos existen.
La imputación es determinista, no una aproximación.

In [ ]:
de = de_raw.copy()

mascara_nulos = de.Delay_End.isna()

# Cast explícito a datetime64[us] para evitar conflicto de precisión con pandas
imputados = (
    de.loc[mascara_nulos, "Delay_Start"]
    + pd.to_timedelta(de.loc[mascara_nulos, "Delay_Hours"], unit="h")
).astype("datetime64[us]")

de.loc[mascara_nulos, "Delay_End"] = imputados

print(f"Delay_Events — imputación de Delay_End:")
print(f"  Registros imputados  : {mascara_nulos.sum()}")
print(f"  Nulos restantes      : {de.Delay_End.isna().sum()}")
print()
print("  Relación Delay_Start + Delay_Hours = Delay_End verificada al minuto en todos los registros completos.")

assumptions_log.append({
    "ID"                    : "A07",
    "Tabla"                 : "Delay_Events",
    "Decisión"              : "120 valores nulos de Delay_End imputados como Delay_Start + Delay_Hours.",
    "Justificación"         : "Relación exacta (diferencia 0.0 min) verificada en todos los registros completos. "
                              "Imputación matemáticamente determinista.",
    "Registros afectados"   : int(mascara_nulos.sum()),
    "Impacto en costo (%)"  : 0.0,
    "Riesgo"                : "Ninguno.",
})

## 5. Quality_Findings

Dos issues:

1. **140 filas con `Work_Order_ID` nulo** — sin atribución posible a una orden de trabajo.
   Algunas tienen `Rework_Hours > 0`; se excluyen y se documenta el costo en horas.

2. **123 filas con `Rework_Flag = "No"` pero `Rework_Hours > 0`** — inconsistencia en el campo flag.
   Representan el 1.4 % de las horas totales de rework.
   Decisión: derivar `has_rework` directamente de `Rework_Hours > 0`.
   Las horas son el campo cuantitativo y son la fuente de verdad; la flag es un campo categórico
   que no fue mantenido correctamente en el sistema de origen.

> **Nota de liderazgo:** la inconsistencia en `Rework_Flag` no es solo un problema de datos —
> puede indicar que el proceso de registro de calidad no está estandarizado entre estaciones o turnos.
> Es un hallazgo de proceso relevante para la sección de recomendaciones.

In [ ]:
qf = qf_raw.copy()

# Eliminar findings sin Work_Order_ID
mascara_huerfanos_qf  = qf.Work_Order_ID.isna()
horas_rework_excluidas = qf.loc[mascara_huerfanos_qf, "Rework_Hours"].sum()

print(f"Quality_Findings — registros sin Work_Order_ID:")
print(f"  Registros eliminados : {mascara_huerfanos_qf.sum()}")
print(f"  Horas de rework excluidas: {horas_rework_excluidas:.1f} h")

qf = qf[~mascara_huerfanos_qf].copy()

# Resolver inconsistencia entre Rework_Flag y Rework_Hours
inconsistencias  = ((qf.Rework_Flag == "No") & (qf.Rework_Hours > 0)).sum()
horas_afectadas  = qf.loc[(qf.Rework_Flag == "No") & (qf.Rework_Hours > 0), "Rework_Hours"].sum()

qf["has_rework"] = qf.Rework_Hours > 0

print()
print(f"Inconsistencia Rework_Flag vs. Rework_Hours:")
print(f"  Flag=No con horas > 0: {inconsistencias} registros  "
      f"({horas_afectadas:.1f} h = {horas_afectadas/qf.Rework_Hours.sum()*100:.1f} % del total)")
print(f"  has_rework=True en dataset limpio: {qf.has_rework.sum():,} registros")
print()
print("  Se ignora Rework_Flag. has_rework se deriva de Rework_Hours > 0.")

assumptions_log.append({
    "ID"                    : "A08",
    "Tabla"                 : "Quality_Findings",
    "Decisión"              : "140 findings sin WO_ID excluidos. has_rework derivado de Rework_Hours > 0 "
                              "(Rework_Flag ignorada).",
    "Justificación"         : "Sin WO_ID no hay atribución posible. Rework_Flag tiene 123 inconsistencias — "
                              "se prioriza el campo numérico como fuente de verdad. "
                              "La inconsistencia puede indicar registro no estandarizado entre estaciones.",
    "Registros afectados"   : int(mascara_huerfanos_qf.sum() + inconsistencias),
    "Impacto en costo (%)"  : 0.0,
    "Riesgo"                : "Bajo — 1 % de findings excluidos; inconsistencia de flag es un "
                              "hallazgo de proceso, no un error de datos.",
})

## 6. Log de supuestos y riesgos

Consolidación de todas las decisiones de limpieza tomadas en esta fase.
Esta tabla alimenta directamente la sección **"Supuestos y riesgos"** del deck ejecutivo.

In [ ]:
log_df = pd.DataFrame(assumptions_log)[[
    "ID", "Tabla", "Decisión", "Justificación",
    "Registros afectados", "Impacto en costo (%)", "Riesgo",
]]

pd.set_option("display.max_colwidth", 90)
log_df

## 7. Exportar datasets limpios

Los datasets se guardan en formato Parquet para mayor eficiencia en la Fase 4 (EDA).
El log de supuestos se exporta también como CSV para reutilizarlo en el deck ejecutivo.

In [ ]:
SALIDA = Path("data/cleaned")
SALIDA.mkdir(parents=True, exist_ok=True)

av.to_parquet(SALIDA / "aircraft_visits.parquet", index=False)
wo_prod.to_parquet(SALIDA / "work_orders_productividad.parquet", index=False)
wo_open.to_parquet(SALIDA / "work_orders_abiertos.parquet", index=False)
lt.to_parquet(SALIDA / "labor_transactions.parquet", index=False)
de.to_parquet(SALIDA / "delay_events.parquet", index=False)
qf.to_parquet(SALIDA / "quality_findings.parquet", index=False)
log_df.to_csv(SALIDA / "log_supuestos.csv", index=False, encoding="utf-8-sig")

print("Datasets exportados correctamente:")
for archivo in sorted(SALIDA.iterdir()):
    tamano = archivo.stat().st_size / 1024
    print(f"  {archivo.name:<50} {tamano:>8.1f} KB")